In [ ]:
# Step 1: Dependencies & Model Setup
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

class XLMRobertaTextClassifier(nn.Module):
    def __init__(self, model_name='cardiffnlp/twitter-xlm-roberta-base-sentiment', num_classes=3):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(768, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        return self.classifier(cls_repr)

In [ ]:
# Step 2: Dataset Simulation across 10 Languages
MULTILINGUAL_TEXT_SAMPLES = [
    {"text": "I love this product, it is absolutely amazing and fantastic!", "label": 2},
    {"text": "Extremely bad quality, broke after one day. Very disappointed.", "label": 0},
    {"text": "The package arrived today in standard condition.", "label": 1},
    {"text": "¡Me encanta este servicio! Es maravilloso y muy eficiente.", "label": 2},
    {"text": "Servicio pésimo, la calidad es horrible y muy mala.", "label": 0},
    {"text": "J'adore ce produit magnifique, excellente qualité !", "label": 2},
    {"text": "Très déçu, produit cassé après la première utilisation.", "label": 0},
    {"text": "Ich liebe dieses Produkt, fantastische Qualität und toll!", "label": 2},
    {"text": "Schlechter Service und minderwertige Qualität. Nie wieder.", "label": 0},
    {"text": "मुझे यह प्रोडक्ट बहुत पसंद आया, यह बेहद शानदार है!", "label": 2},
    {"text": "बहुत ही ख़राब सर्विस, क्वालिटी बहुत बुरी है।", "label": 0},
    {"text": "இந்த தயாரிப்பு மிகவும் அற்புதம், எனக்கு மிகவும் பிடித்துள்ளது!", "label": 2},
    {"text": "மிகவும் மோசமான தரம், வாங்கிய உடனே பழுதாகிவிட்டது.", "label": 0},
    {"text": "我非常喜欢这个产品，质量太棒了！", "label": 2},
    {"text": "这个产品太糟糕了，完全无法使用。", "label": 0},
    {"text": "この商品は本当に素晴らしいです！とても満足しています。", "label": 2},
    {"text": "أنا أحب هذا المنتج الرائع جداً!", "label": 2},
    {"text": "Мне очень нравится этот отличный товар!", "label": 2}
]

class SyntheticTextDataset(Dataset):
    def __init__(self, tokenizer, num_samples=1200):
        base_len = len(MULTILINGUAL_TEXT_SAMPLES)
        samples = [MULTILINGUAL_TEXT_SAMPLES[i % base_len] for i in range(num_samples)]
        texts = [s['text'] for s in samples]
        encoded = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
        self.input_ids = encoded['input_ids']
        self.attention_mask = encoded['attention_mask']
        self.labels = torch.tensor([s['label'] for s in samples], dtype=torch.long)
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'label': self.labels[idx]
        }

model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
tokenizer = AutoTokenizer.from_pretrained(model_name)25
dataset = SyntheticTextDataset(tokenizer, num_samples=1200)

train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
# Step 3: Train & Validation Loop with Metric Tracking
model = XLMRobertaTextClassifier().to(device)
optimizer = AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

epochs = 6
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, epochs + 1):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        y = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * y.size(0)
        correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total += y.size(0)
        
    tr_loss = t_loss / total
    tr_acc = (correct / total) * 100
    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    
    # Validation
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            y = batch['label'].to(device)
            logits = model(ids, mask)
            loss = criterion(logits, y)
            v_loss += loss.item() * y.size(0)
            v_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            v_total += y.size(0)
            
    va_loss = v_loss / v_total
    va_acc = (v_correct / v_total) * 100
    val_losses.append(va_loss)
    val_accs.append(va_acc)
    
    print(f"Epoch [{epoch}/{epochs}] - Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}% | Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.2f}%")

In [ ]:
# Step 4: Plot Training & Validation Graphs and Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graph 1: Loss Curve
axes[0].plot(range(1, epochs + 1), train_losses, label='Train Loss', marker='o', color='#ef4444')
axes[0].plot(range(1, epochs + 1), val_losses, label='Validation Loss', marker='s', color='#f59e0b')
axes[0].set_title('Text Model: Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# Graph 2: Accuracy Curve
axes[1].plot(range(1, epochs + 1), train_accs, label='Train Accuracy', marker='o', color='#3b82f6')
axes[1].plot(range(1, epochs + 1), val_accs, label='Validation Accuracy', marker='s', color='#10b981')
axes[1].set_title('Text Model: Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

# Step 5: Test Set Confusion Matrix & Metrics
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        y = batch['label'].to(device)
        logits = model(ids, mask)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Text Model Test Set Confusion Matrix', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("TEXT MODEL CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=['Negative', 'Neutral', 'Positive']))